# Lesson 2 — Linear Programming Fundamentals

**Track:** Basic &nbsp;|&nbsp; **Estimated time:** 2 hours

## Learning objectives

1. Write any LP in standard form.
2. Describe the geometry of the feasible polyhedron and why optima sit at vertices.
3. State the simplex method at a conceptual level (no proof).
4. Recognize the LP dual and state weak/strong duality.
5. Use `discopt`'s LP solver and inspect the basis.

## 1. Standard form

The standard-form LP {cite:p}`Bertsimas1997,Dantzig1963` is:

$$
\min_{x} \; c^\top x \quad \text{s.t.} \quad Ax = b, \; x \ge 0
$$

with $A \in \mathbb{R}^{m \times n}$, $b \in \mathbb{R}^m$, $c \in \mathbb{R}^n$. Any LP can be converted: inequalities $a^\top x \le b$ become $a^\top x + s = b, s \ge 0$ with a *slack* variable; free variables $x$ become $x^+ - x^-$ with $x^+, x^- \ge 0$.

## 2. Geometry

The feasible set $\{x : Ax = b, x \ge 0\}$ is a **polyhedron**. Three facts:

- **Vertices = basic feasible solutions (BFS).** A BFS picks $m$ of the $n$ columns of $A$ to be the "basis"; the remaining variables are 0.
- **Fundamental theorem of LP** {cite:p}`Bertsimas1997`: if an LP has a finite optimum, at least one optimum is at a vertex.
- **Bounded vs unbounded vs infeasible.** Three possible outcomes; a solver must distinguish them.

## 3. Simplex, in one paragraph

Start at any BFS. Examine adjacent vertices (those differing by one basis swap). If any adjacent vertex has lower cost, move there; otherwise, current vertex is optimal. The algorithm is **finite** under anti-cycling rules like Bland's {cite:p}`Bland1977`. In the worst case it is exponential ({cite:t}`KleeMinty1972` constructed a cube where simplex visits all $2^n$ vertices), but in practice it is fast on most LPs.

## 4. Duality

Every LP has a **dual**. For the primal $\min c^\top x$ s.t. $Ax \ge b, x \ge 0$, the dual is

$$
\max_{y \ge 0} \; b^\top y \quad \text{s.t.} \quad A^\top y \le c.
$$

Dual variables $y$ are **shadow prices**: $y_i$ = marginal change in optimal cost per unit relaxation of constraint $i$.

- **Weak duality:** $b^\top y \le c^\top x$ for any primal-feasible $x$, dual-feasible $y$.
- **Strong duality:** if either has a finite optimum, both do, and the values are equal.
- **Complementary slackness:** at optimum, $y_i \cdot (a_i^\top x - b_i) = 0$ for all $i$.

We use this constantly: feasible duals certify lower bounds; complementary slackness identifies which constraints are binding.

In [ ]:
import numpy as np
import discopt as do

# Primal: min c'x  s.t.  Ax >= b, x >= 0
c = np.array([3.0, 2.0])
A = np.array([[1, 1], [2, 1], [1, 3]])
b = np.array([4.0, 5.0, 6.0])

m = do.Model("lp")
x = m.add_variables(2, lb=0, name="x")
m.add_constraints(A @ x >= b)
m.minimize(c @ x)
res = m.solve()
print("primal optimum :", res.objective, "at x =", res.x)
print("dual y         :", res.dual)              # shadow prices
print("active constr. :", np.where(res.dual > 1e-8)[0])


## 5. Verifying duality numerically

The dual we just printed should satisfy three conditions:

1. **Dual feasibility:** $A^\top y \le c$.
2. **Equal objectives:** $b^\top y = c^\top x^\star$.
3. **Complementary slackness:** $y_i = 0$ on slack constraints, $y_i > 0$ on tight constraints (under non-degeneracy).

These are exactly the **KKT conditions** specialized to LP, which we revisit for general NLP in Lesson 9 {cite:p}`Boyd2004,Nocedal2006`.

## 6. The two ways LPs go wrong

| Pathology | Symptom | Diagnosis |
|-----------|---------|-----------|
| **Degeneracy** | Multiple BFS map to same vertex; simplex may stall | Use Bland's rule {cite:p}`Bland1977` |
| **Numerical** | Bad scaling; ill-conditioned basis | Rescale, use IPM (Lesson 12) |

The presolve step, which we cover in Lesson 15 {cite:p}`Savelsbergh1994`, eliminates many of these issues automatically.

## References
{cite:p}`Bertsimas1997,Dantzig1963,Bland1977,Boyd2004` for the algorithmic side; {cite:p}`Williams2013` for modeling style.